# Stage 9c — Modèle recalibré & nouvelle soumission

On applique la correction validée en 9b : **augmentation réaliste + pénalité classe 0 (poids 2.6)**, ensemble HGB+LR, seuil 0.5. Écrit `submission_v3.csv`.

In [1]:
import numpy as np, pandas as pd, warnings
warnings.filterwarnings("ignore")
TRAIN_PATH="Train.csv"; TEST_PATH="Test.csv"; SAMPLE_PATH="SampleSubmission.csv"
NODATA=-9999
MONTHS=[f"{i:02d}" for i in range(1,13)]
BANDS=["VH","VV","blue","green","nir","nira","re1","re2","re3","red","swir1","swir2"]
# profil de donnees manquantes MESURE sur le vrai test (Stage 8)
WINTER_BIAS=np.array([.88,.78,.62,.50,.41,.43,.36,.41,.50,.80,.75,.87])

def load_raw(p): return pd.read_csv(p).replace(NODATA, np.nan)
def safe_ratio(a,b): return (a-b)/(a+b)

def build_features(df):
    feats={}; idx={}
    for m in MONTHS:
        g=df[f"green_{m}"]; n=df[f"nir_{m}"]; r=df[f"red_{m}"]; s1=df[f"swir1_{m}"]
        idx.setdefault("NDWI",[]).append(safe_ratio(g,n))
        idx.setdefault("MNDWI",[]).append(safe_ratio(g,s1))
        idx.setdefault("NDVI",[]).append(safe_ratio(n,r))
        idx.setdefault("NDMI",[]).append(safe_ratio(n,s1))
        idx.setdefault("VHmVV",[]).append(df[f"VH_{m}"]-df[f"VV_{m}"])
    def summ(monthly,name):
        a=np.column_stack([np.asarray(x,float) for x in monthly])
        feats[f"{name}_mean"]=np.nanmean(a,1); feats[f"{name}_std"]=np.nanstd(a,1)
        feats[f"{name}_min"]=np.nanmin(a,1);  feats[f"{name}_max"]=np.nanmax(a,1)
        feats[f"{name}_range"]=np.nanmax(a,1)-np.nanmin(a,1); feats[f"{name}_med"]=np.nanmedian(a,1)
        feats[f"{name}_nvalid"]=np.isfinite(a).sum(1)
    for b in BANDS: summ([df[f"{b}_{m}"] for m in MONTHS], b)
    for k,v in idx.items(): summ(v,k)
    return pd.DataFrame(feats,index=df.index).replace([np.inf,-np.inf],np.nan)

def corrupt_realistic(df, seed=0):
    rng=np.random.RandomState(seed); d=df.copy()
    cols={b:[f"{b}_{m}" for m in MONTHS] for b in BANDS}
    arr={b:d[cols[b]].values.astype(float) for b in BANDS}
    for i in range(len(d)):
        miss=rng.rand(12)<WINTER_BIAS*rng.uniform(0.7,1.1)
        for b in BANDS: arr[b][i,miss]=np.nan
    for b in BANDS: d[cols[b]]=arr[b]
    return d

In [2]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GroupKFold
from sklearn.cluster import KMeans
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score, confusion_matrix

def make_hgb(**kw):
    return HistGradientBoostingClassifier(max_iter=400, learning_rate=0.05,
            max_leaf_nodes=7, l2_regularization=1.0, random_state=42, **kw)
def make_lr(**kw):
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                         LogisticRegression(max_iter=2000, **kw))

def fit_ensemble(train_df, ytr, X_eval, class_weight=None, n_corrupt=2):
    """Augmente (realiste) + ensemble HGB+LR. class_weight={0:w,1:1} penalise les faux positifs."""
    parts=[train_df]+[corrupt_realistic(train_df,seed=i+1) for i in range(n_corrupt)]
    aug=pd.concat(parts,ignore_index=True); yaug=np.concatenate([ytr]*len(parts))
    Xa=build_features(aug)
    h=make_hgb(class_weight=class_weight).fit(Xa,yaug)
    l=make_lr(class_weight=class_weight).fit(Xa,yaug)
    return 0.5*h.predict_proba(X_eval)[:,1]+0.5*l.predict_proba(X_eval)[:,1]

In [3]:
train=load_raw(TRAIN_PATH); test=load_raw(TEST_PATH); sample=pd.read_csv(SAMPLE_PATH)
y=train["label"].values
X_test=build_features(test)

CLASS_WEIGHT={0:2.6, 1:1.0}   # correction validee en Stage 9b
proba=fit_ensemble(train, y, X_test, class_weight=CLASS_WEIGHT)

submission=pd.DataFrame({"ID":test["ID"],
                         "TargetF1":(proba>=0.5).astype(int),
                         "TargetRAUC":proba})
checks={"cols":list(submission.columns)==list(sample.columns),
        "rows":len(submission)==len(sample),
        "ids":set(submission.ID)==set(sample.ID),
        "f1_binaire":set(submission.TargetF1.unique()).issubset({0,1}),
        "rauc_0_1":submission.TargetRAUC.between(0,1).all(),
        "no_nan":submission.notna().all().all()}
for k,v in checks.items(): print(("OK " if v else "ERR"),k)
assert all(checks.values())
print(f"\nbassins predits: {submission.TargetF1.sum()} ({submission.TargetF1.mean():.1%}) vs train {y.mean():.1%}")
print("(avant correction: ~55%. Plus proche de 40% = mieux calibre.)")
submission.to_csv("submission_v3.csv", index=False)
print("ecrit: submission_v3.csv")
submission.head()

OK  cols
OK  rows
OK  ids
OK  f1_binaire
OK  rauc_0_1
OK  no_nan

bassins predits: 528 (51.3%) vs train 40.4%
(avant correction: ~55%. Plus proche de 40% = mieux calibre.)
ecrit: submission_v3.csv


,ID,TargetF1,TargetRAUC
0,ID_TS_NEW_SBZAYD5I,1,0.998573
1,ID_TS_NEW_7SPRN3PB,1,0.996999
2,ID_TS_NEW_LZOWPHLC,1,0.573566
3,ID_TS_NEW_DN6TUF64,0,0.461735
4,ID_TS_NEW_95N82M49,0,0.000137


## Récap de la démarche complète d'amélioration
1. **Diagnostic (9a)** : F1 bridé par une sur-prédiction des bassins (trop de faux positifs au seuil 0.5).
2. **Tests (9b)** : pénaliser la classe 0 (poids ~2.6) corrige ça, confirmé en CV honnête (F1 0.90→0.91, blend 0.93→0.94).
3. **Soumission (9c)** : modèle recalibré, taux prédit ramené vers le vrai 40%.

Après soumission : compare à 0.819. Si ça monte, le diagnostic faux-positifs était le bon. Et toujours : **une soumission, puis retour au CV honnête** avant la suivante.